# Interpreting Features
You've already gained experienced implementing methods that identify features, like SNMF and SAEs. In this practicum, you will focus on interpreting features learned by an SAE and understanding what they represent.

Interpreting SAEs involves producing natural-language descriptions of the features they discover, like punctuation styles or religions. There are multiple ways for doing this, but the prevalent approach is *input-centric*, where features are described based on the inputs that most strongly activate them.

But this raises several questions:

- How can you be confident that you've gathered an appropriate and representative set of inputs for evaluating a feature?

- Once you have a set of activating inputs, how do you turn that into a coherent, human-interpretable description of the feature?

- How do you guarantee that the feature descriptions you created from the activating inputs align with how the feature causally influences the model's behavior?


Another limitation is that some SAE features may appear dead (they don't activate on any inputs) or are otherwise uninterpretable when examined purely through *input-centric* methods. In this assignment, you will generate descriptions of your SAE's features, identify dead or inactive features, and attempt to interpret these dead features using *output-centric* techniques that focus on the features' effects on the model's output.

If you'd like to dive more into analyzing features in common SAEs, you should check out [Neuronpedia's](https://www.neuronpedia.org/) useful interface.

> *Note: The methods and challenges examined in this practicum apply not only to SAEs, but also to features discovered using other techniques.*

## Practicum Goal  
In this practicum, you'll learn how to interpret the features learned by SAEs using both *input-centric* and *output-centric* techniques.

---

## The Challenge  
Choose a feature that is difficult to interpret and use alternative interpretability techniques that you are already familiar with (e.g. Patchscopes) to interpret them.

## Setup
Run this cell to load dependencies and utility modules from a local folder in this project (prefers `snmf/`, with fallback to `snmf-mlp/` if needed).
The cell downloads `concept_train.json` into the local repo data directory so the dataset loader can run without Colab-specific paths.
Upload your SAE's `.pt` file and update the local path name in `model_path` if you are using a custom checkpoint.
Set the other parameters to match those used to train your SAE.
> *Note: If you get "MessageFactory" warnings, re-run the cell and it should execute normally.*


In [1]:
import sys
import math
from pathlib import Path

import requests
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from typing import Tuple
from huggingface_hub import hf_hub_download

# Resolve project root from common notebook working directories.
cwd = Path.cwd().resolve()
project_root = None
for candidate in (cwd, cwd.parent, cwd.parent.parent):
    if (candidate / "snmf").exists():
        project_root = candidate
        break
if project_root is None:
    raise FileNotFoundError("Could not find project root containing the 'snmf' folder.")

# Prefer local modules under snmf/ and fall back to snmf-mlp/ if needed.
repo_candidates = [
    project_root / "snmf" / "snmf-mlp-decomposition",
    project_root / "snmf",
    project_root / "snmf-mlp",
]
local_repo_dir = None
for candidate in repo_candidates:
    if (candidate / "data_utils").exists() and (candidate / "llm_utils").exists():
        local_repo_dir = candidate
        break
if local_repo_dir is None:
    raise FileNotFoundError(
        "Could not find local repo modules (expected data_utils and llm_utils) under snmf/ or snmf-mlp/."
    )

if str(local_repo_dir) not in sys.path:
    sys.path.append(str(local_repo_dir))

from data_utils.concept_dataset import SupervisedConceptDataset
from llm_utils.activation_generator import ActivationGenerator, extract_token_ids_sample_ids_and_labels

concept_data_path = local_repo_dir / "data" / "concept_train.json"
concept_data_path.parent.mkdir(parents=True, exist_ok=True)

response = requests.get(
    "https://huggingface.co/datasets/dhgottesman/sae_concept/resolve/main/concept_train.json"
)
response.raise_for_status()
with open(concept_data_path, "wb") as f:
    f.write(response.content)

class SAE(nn.Module):
    """
    Sparse Autoencoder

    Encoder:  a = W_e (x - b_d) + b_e
    Decoder:  x_hat = W_d^T a + b_d

    Forward returns:
        recon, activations   # shapes: (batch_size, input_dim), (batch_size, hidden_dim)

    Args
    ----
    input_dim : int
        input dimensionality
    hidden_dim : int
        number of dictionary atoms (latent units)
    eps : float
        Numerical stability for normalization.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        l1_lambda: float,
        eps: float = 1e-8,
    ) -> None:
        super().__init__()
        self.l1_lambda = l1_lambda
        self.eps = float(eps)

        self.encoder = nn.Linear(input_dim, hidden_dim, bias=True)
        self.decoder = nn.Linear(hidden_dim, input_dim, bias=True)

        self.init_parameters()

    def init_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.encoder.weight, a=math.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.encoder.weight)
        bound = 1 / math.sqrt(fan_in)
        nn.init.uniform_(self.encoder.bias, -bound, bound)

        nn.init.xavier_uniform_(self.decoder.weight)
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.decoder.weight)
        bound = 1 / math.sqrt(fan_in)
        nn.init.uniform_(self.decoder.bias, -bound, bound)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Return activations a (after ReLU nonlinearity), shape (B, H)."""
        a = self.encoder(x - self.decoder.bias)
        return F.relu(a)

    def decode(self, a: torch.Tensor) -> torch.Tensor:
        """Decode activations to reconstruction, shape (B, D)."""
        W = self._normalize_columns(self.decoder.weight)
        return F.linear(a, W, self.decoder.bias)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
          x: (batch_size, input_dim)
        Returns: (recon, a)
            recon: the reconstructed activations `x_hat` (batch_size, input_dim)
            a: the encoder feature activations `a` (batch_size, hidden_dim)
        """
        a = self.encode(x)
        recon = self.decode(a)
        return recon, a

    def _normalize_columns(self, W: torch.Tensor) -> torch.Tensor:
        norms = torch.linalg.norm(W, dim=0, keepdim=True).clamp_min(self.eps)
        return W / norms

def _id_to_str(tokenizer_or_model, token_id: int):
    """
    Minimal helper that works with either:
      - HookedTransformer-like objects exposing .to_str_tokens(LongTensor)->List[str]
      - HF tokenizers exposing .convert_ids_to_tokens(int)->str
    """
    if hasattr(tokenizer_or_model, "to_str_tokens"):
        ids = torch.tensor([token_id], dtype=torch.long, device="cpu")
        return tokenizer_or_model.to_str_tokens(ids)[0]
    if hasattr(tokenizer_or_model, "convert_ids_to_tokens"):
        return tokenizer_or_model.convert_ids_to_tokens(int(token_id))
    return str(token_id)

def generate_token_contexts(tokens, sample_ids, act_generator):
    context_window = 15

    token_ds = []
    for i in range(len(tokens)):
        current_sample_id = sample_ids[i]
        token_str = act_generator.model.to_str_tokens([tokens[i]])[0][0]

        start = max(0, i - context_window)
        end = min(len(tokens), i + context_window + 1)

        context_tokens = [
            act_generator.model.to_str_tokens([tokens[j]])[0][0] for j in range(start, end) if sample_ids[j] == current_sample_id
        ]

        context_str = "".join(context_tokens)
        token_ds.append((token_str, context_str))

    return token_ds


/opt/anaconda3/envs/snmf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset and Dataloader (Recap)
**Dataset** The raw dataset consists of a collection of sentences. Each sentence is tokenized and processed by the model, which has a hidden dimension of `input_dim`.
For every token, activations are extracted from the specified `layer` and `factorization_mode` (e.g., the residual stream at layer 30), which were used to train the SAE.
The resulting dataset aggregates activations across all tokens. Specifically, if tokenizing the full corpus produces a total of `N` tokens, the dataset can be represented as a tensor of shape
`(N, input_dim)`.

**Dataloader** During training, the dataset is batched to form a data loader, where each batch has shape
`(batch_size, input_dim`, yielding `N/batch_size` batches in total.

**Other parameters** `hidden_dim` is the number of features in the SAE, and `l1_lambda` controls the strength of the L1 loss during SAE training.

*The default values are set to match the SAE provided in this practicum. If you wish to use your own trained SAE, adjust the parameters accordingly.*


In [72]:

# Device to load data to, default is CPU and only factorization and generation occurs on GPU
device = "mps" if torch.cuda.is_available() else "cpu"
data_device = 'cpu'
data_path = str(concept_data_path)
dataset = SupervisedConceptDataset(data_path)

model_name = "gpt2-large"
# Layers of model to inspect
layer = 30 # @param {type:"integer"}
# Device to load model to for generation
model_device = device
# factorization mode, factorize residual or mlp layers
factorization_mode = 'residual'  # @param ["residual", "mlp"]

# Library for extracting the activations from a specific layer and factorization mode.
# Returns a list of activations with each entry corresponding respectively to the layers indicated.
# Each element is a tensor of shape B x D, where B is the number of samples and D is the hidden dimension of the model.
act_generator = ActivationGenerator(model_name, model_device=model_device, data_device=data_device, mode=factorization_mode)
activations, freq = act_generator.generate_multiple_layer_activations_and_freq(dataset, [layer])
tokens, sample_ids, _ = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

token_ds = generate_token_contexts(tokens, sample_ids, act_generator)

batch_size = 256
loader = DataLoader(activations[0], batch_size=batch_size, shuffle=False, pin_memory=False)

input_dim = 1280 # @param {type:"integer"}
hidden_dim = 500 # @param {type:"integer"}
l1_lambda = 1e-3 # @param {type:"number"}

repo_id = "dhgottesman/Concept_SAE_GPT2Large_HD500_L30"
filename = "sae_l30_H500_keep64_value.pt"

ckpt_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
)

sae = SAE(input_dim=1280, hidden_dim=hidden_dim, l1_lambda=1e-3).to(model_device)
state_dict = torch.load(ckpt_path, map_location=model_device)
sae.load_state_dict(state_dict["state_dict"])

Loaded pretrained model gpt2-large into HookedTransformer


Extracting token IDs: 100%|██████████| 56/56 [00:00<00:00, 1122.21it/s]


<All keys matched successfully>

In [73]:
activations[0].shape

torch.Size([26413, 1280])

## *Input Centric*: Top Activating Tokens
In this method, SAE features are interpreted by examining the tokens in the dataset that produce the strongest activations for a given feature.

For example, in this Neuronpedia [feature](https://www.neuronpedia.org/gpt2-small/6-res_scefr-ajt/650), the highest-activating tokens include ` long`, ` high`, ` wide`, and ` tall` when they appear in measurement-related contexts. Based on this consistent pattern, the feature is interpreted as representing *"measurements in meters or feet"*.

## 📝 Exercise: Implementing Top Activating Tokens
In this exercise, you’ll complete the implementation of the `get_top_activating_indices` function. The goal is to identify which tokens in the dataset most strongly activate a specific SAE feature identified by `feature_idx`.

For each batch in `loader`:

1. Run the SAE encoder to obtain the feature activations for all input tokens in the dataset.
2. Select only the activations for the `feature_idx` feature which is a tensor of shape `(batch_size,)`.
Append this tensor to `activations_list`.

3. After iterating over all batches, `activations_list` will contain many tensors of shape `(batch_size,)`.
Concatenate them into a single tensor called `activations` of shape `(N,)`. Each value in the `activations` tensor indicates how strongly the corresponding input token activated this feature.

4. *Provided for you!* Create a boolean mask to filter activations greater than a `minimum_activation` threshold.

    This mask is applied to:

    - The activation tensor `activations`.
    - The index tensor `idxs` representing the indexes of tokens in the dataset.
      We mask this tensor so that we can track the indices of the tokens that sufficiently activate the feature.

5. From the filtered activations generated in step 4, keep only the `idxs` and corresponding `activations` of the tokens which yield the largest `k` activations across the entire dataset.

Return:
- The indices of the top-k activating tokens `top_idxs`.
- Their corresponding activation values `top_activations`.


In [74]:


@torch.no_grad()
def get_top_activating_indices(
    sae, loader, feature_idx, k=10, minimal_activation=0.0, device=None
):
    """
    Iterate the `loader`, get SAE activations, and return the top dataset
    indices for the given latent unit (feature_idx).
    """
    sae.eval()
    sae.to(device)

    activations_list = []

    # Extracting Feature Activations
    for x in loader:
        x = x.to(device) # (batch_size, d_model)
        a = sae.encode(x) # (batch_size, feature_dim)
        feature_activations = a[:, feature_idx] # (batch_size,)
        activations_list.append(feature_activations)

    activations = torch.cat(activations_list)  # (N,)
    idxs = torch.arange(activations.shape[0], dtype=torch.long)

    # Filter by minimal_activation
    mask = activations > float(minimal_activation)
    if not mask.any().item():
        return [], []

    activations = activations[mask]
    idxs = idxs[mask]

    # Applying Top-k Filtering
    kk = min(k, activations.shape[0])
    _, top_pos = torch.topk(activations, k=kk)
    top_activations = activations[top_pos]
    top_idxs = idxs[top_pos]

    return top_idxs.tolist(), top_activations.tolist()


## Dead features

In [75]:
@torch.no_grad()
def get_feature_activations(sae, loader) -> torch.Tensor:
    """
    Iterate the `loader`, get SAE activations, and return a tensor of shape (N, feature_dim)
    containing the feature activations for all samples.
    """
    sae.eval()
    sae.to(device)

    activations_list = []

    for x in loader:
        x = x.to(device) # (batch_size, d_model)
        a = sae.encode(x) # (batch_size, feature_dim)
        activations_list.append(a.cpu())

    activations = torch.cat(activations_list)  # (N, feature_dim)
    return activations

@torch.no_grad()
def dead_features(feature_activations: torch.Tensor, threshold: float = 0.0) -> Tuple[int, torch.Tensor]:
    """
    Identify dead features that never activate above the given threshold.
    Args:
        - feature_activations: Tensor of shape (N, feature_dim) containing activations for all samples.
        - threshold: Activation threshold to determine if a feature is active.
    Returns:
        - num_dead_features: The count of dead features.
        - dead_feature_indices: Tensor containing indices of dead features.
    """
    # Check if any activation for each feature exceeds the threshold
    feature_dim = feature_activations.shape[1]
    active_mask = (feature_activations > threshold).any(dim=0)  # (feature_dim,)
    assert active_mask.shape[0] == feature_dim, "Active mask should have the same length as feature dimension."
    dead_feature_indices = torch.where(~active_mask)[0]  # Indices of dead features
    num_dead_features = dead_feature_indices.shape[0]
    return num_dead_features, dead_feature_indices

## Retrieve the Top-$k$ activating tokens for the SAE features.
Print the top activating tokens for every concept in the dimension of your SAE i.e. if your SAE has a hidden dimension of 500, you can do this for all 500 features.


In [76]:
for feature_idx in range(10):
    top_indices, top_activations = get_top_activating_indices(sae, loader, feature_idx=feature_idx, k=10, device=model_device)
    top_activations = [
        {'token': token_ds[i][0], 'activation': a, 'context': token_ds[i][1]}
        for i, a in zip(top_indices, top_activations)
    ]
    print(f"###########{feature_idx}#############\n")
    # Iterate over the top_activations and print the token, activation value, and context for each top activating index.
    for activation in top_activations:
        print(f"{activation['token']}\t\t{activation['activation']}\t\t{activation['context']}")

###########0#############

's		11.717001914978027		 the themes of jealousy and manipulation.
6. A Midsummer Night's Dream, while enchanting with its magic, deals with the complexities of love
 and		10.321419715881348		1. "The Old Man and the Sea" by Ernest Hemingway  
2. "A
ru		9.366202354431152		 year, particularly around the 20th or 21st of March, during Nowruz celebrations. The inclusion of these symbols, with their associations to themes of
 in		8.39058780670166		 elevated Seattle's profile, with "Singles" and "Sleepless in Seattle" often compared in their portrayal of the city. "Singles,"
lee		7.362909317016602		 several films elevated Seattle's profile, with "Singles" and "Sleepless in Seattle" often compared in their portrayal of the city. "Sing
 Kar		6.973941802978516		The founder of House Karstark is Rickard Karstark, who established his lineage during a time when local resources were essential
lee		4.838860034942627		 classified as a romantic comedy, captures the grunge scene

## Exercise: Exploring Feature Concepts and Dead Features
Examine the tokens that most strongly activate each feature in your SAE and reflect on the following:

1. **What kinds of concepts do these features seem to represent?** Are they syntactic (e.g., punctuation), semantic (e.g., topics, entities) or something else?
2. **Are there any features that appear difficult to interpret or that seem "dead"?**
For example, do some features activate on tokens that don't form a clear pattern or concept or on very few inputs?
*Note these features, as we will take a stab at interpretting them next 🙏*


In [77]:
# Let's print out all dead features
feature_activations = get_feature_activations(sae, loader)
num_dead_features, dead_feature_indices = dead_features(feature_activations, threshold=0.0)
print(f"Number of dead features: {num_dead_features}")
dead_feature_str = ", ".join(str(idx) for idx in dead_feature_indices.tolist())
dead_feature_str

Number of dead features: 27


'5, 9, 13, 19, 20, 27, 50, 114, 122, 144, 158, 165, 185, 188, 201, 239, 279, 283, 368, 369, 373, 420, 426, 432, 445, 460, 464'

#### List of dead feature indices
These features have no activating tokens in the dataset and they are 27 in number.

'5, 9, 13, 19, 20, 27, 50, 114, 122, 144, 158, 165, 185, 188, 201, 239, 279, 283, 368, 369, 373, 420, 426, 432, 445, 460, 464'

#### Other interpretable features
- Feature 1 is completely activated by `?`.
- Feature 2 is activated by `Two` with context containing the phrase `Two Towers`.

Many of the features are polysemantic activated by different tokens and different contexts about different concepts/topics/entities.

## Limitations of *Input-Centric* Methods

A key limitation of analyzing SAE features using only input-activating tokens is that the SAE is trained with an external objective, so the SAE features aren't necessarily aligned with the internal features of the base model. As a result, these learned features may not reflect concepts that affect the model's predictive behavior as expected. To summarize, some of the limitations with input-centric approaches are:

- **Misleading or inconsistent interpretations**  
- **"Dead" features** — appearing uninterpretable simply because they don't activate on the chosen dataset  
- **Dataset dependence** — different datasets yield different activating tokens, changing what concept the feature appears to represent  

*Therefore, feature concepts should be defined by their causal effect on the model's output.*

## *Output-Centric*: Vocabulary Projection

To ground our interpretations in the *causal effect* of features, we will use two output-centric interpretability methods:

1. **Projecting the feature vector into vocabulary space**  
   This allows us to view which tokens the feature is related to in the vocabulary space.

2. **Feature steering and observing output shifts**  
   We will amplify the feature and measure which token probabilities change the most — causally revealing which output tokens are most promoted or suppressed by the feature.

## Exercise: Vocabulary Projection
Your task is to identify a feature that is *"dead"* or uninterpretable according to the top-activating token technique above, and see if you can *"bring it to life"* using the vocabulary projection technique described below.
After you identify this feature, set `feature_idx` accordingly to interpret it.

---

In the previous practicum, we explored the autoencoder architecture of SAEs and learned that the decoder consists of a **dictionary matrix**:

$$
D \in \mathbb{R}^{d \times k} = [\mathbf{d}_0, \mathbf{d}_1, \ldots, \mathbf{d}_k]
$$

where:
- $d$ is the dimension of the input (e.g., the hidden dimension of the base model).
- $k$ is the number of learned SAE features.
- Each column $\mathbf{d}_i$ represents how feature $i$ is encoded in the input space.

Since your SAE was trained to reconstruct activations from the residual stream (or from MLP outputs) both of which live in $\mathbb{R}^d$—and we already saw the effectiveness of LogitLens—it is reasonable to interpret a feature by projecting its dictionary vector $\mathbf{d}_i$ onto the vocabulary space.

For $\mathbf{d}_i$, we compute:

$$
\mathbf{w} = W_U \cdot \text{LayerNorm}(\mathbf{d}_i)
\quad \text{where} \quad \mathbf{w} \in \mathbb{R}^{|\mathcal{V}|}
$$

Here:
- $\mathcal{V}$ is the token vocabulary.
- $W_U \in \mathbb{R}^{|\mathcal{V}| \times d}$ is the base model's unembedding matrix.
- $\mathbf{w}$ is the tensor of logits (scores) for each token in the vocabulary.


We can now examine the top-scoring tokens in $\mathbf{w}$, and interpret them as those most promoted by the feature.

Simply set `feature_idx` to the feature of interest, and run this cell to interpret it.


In [78]:
### 📌 Feature Interpretation via Vocabulary Projection

@torch.no_grad()
def get_vocab_proj(A: torch.Tensor, top_k: int = 50) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Project a direction to vocab logits and return top-k values and indices.
    """
    # move A to model device
    model = act_generator.model
    device = next(model.parameters()).device
    A = A.to(device)
    direction = model.ln_final(A)
    vocab_proj = model.unembed(direction)
    values, indices = torch.topk(vocab_proj, k=top_k)
    return values, indices


feature_idx = 49 # @param {type:"integer"}
top_k = 10
feature_vec = sae.decoder.weight[:,feature_idx]
pos_vals, pos_idx = get_vocab_proj(feature_vec, top_k=top_k)
pos_vals_list = pos_vals.tolist()

# Convert token IDs to strings; ensure int() cast for safety
pos_tok_list = [act_generator.model.to_str_tokens([tid])[0] for tid in pos_idx]
print(f"########{feature_idx}########")
pos_tok_list = [token[0] for token in pos_tok_list]
# Convert list of tokens to a single string for better readability
pos_tok_str = "".join(pos_tok_list)
print(f"Top {top_k} tokens: {pos_tok_str}")

########49########
Top 10 tokens:  solarSolar eclipse eclips Solar telescopes flares telescope orbitsclipse


In [79]:
sae.decoder.weight.shape

torch.Size([1280, 500])

## *Output-Centric*: Causal Intervention

*The following exercises are optional, but you are highly encouraged to try them out at home.*

In the following experiments we will intervene in the model's computation and amplify the feature.
We intervene in the same way we did with SNMF in Practicum 5. Specifically, in the imported `Intervener` class, we register a hook on the component that the SAE was trained on which does the following:
1. Normalizes the feature vector $\mathbf{d}_i$ direction so it has unit length. *This is intrinsic since the dictionary columns are already normalized in our SAE.*
2. Adds the scaled direction $\alpha * \mathbf{d}_i$ to the existing activation (MLP output or residual stream).

---

After registering the hook, we will analyze:
1. **Change in Logit Values** Extract the tokens whose logits in the model's output were most affected by amplifying the feature.
2. **Steering** Observe the effect of amplifying the feature on the model's prediction.

## Exercise: Change in Logit Values
Your task is to implement the `logit_diff` function which takes as input the `base_logits` computed in (1)
and the `intervened_logits` in step (2), and computes the logit difference (3).
Then extract the `top_k` tokens with the largest positive and negative logit value changes.

**Do these results align with the concepts you discovered through the vocabulary projection technique?**


In [80]:

def logit_diff(base_logits, intervened_logits, top_k=10):
    """
    Computes the top positive and negative logit differences between two
    logit tensors for the final token position.

    Args:
        base_logits (torch.Tensor): Logits before intervention.
            Shape: `(1, seq_len, vocab_size)`.
        intervened_logits (torch.Tensor): Logits after intervention.
            Shape: `(1, seq_len, vocab_size)`.
        top_k (int, optional): Number of highest and lowest logit differences to display.
            Defaults to 10.

    Returns:
        Tuple:
            - pos_vals (torch.Tensor): Top `top_k` positive logit differences.
            - pos_idx (torch.Tensor): Corresponding token indices.
            - neg_vals (torch.Tensor): Top `top_k` negative logit differences.
            - neg_idx (torch.Tensor): Corresponding token indices.
    """
    # Ensure input shapes are correct
    assert base_logits.shape == intervened_logits.shape, "Input logits must have the same shape."
    assert base_logits.dim() == 3, "Input logits must be 3-dimensional (1, seq_len, vocab_size)."

    # Compute logit differences for the final token position
    logit_diff = intervened_logits[0, -1] - base_logits[0, -1]  # Shape: (vocab_size,)

    # Get top-k positive and negative logit differences
    pos_vals, pos_idx = torch.topk(logit_diff, k=top_k, largest=True)
    neg_vals, neg_idx = torch.topk(logit_diff, k=top_k, largest=False)

    return pos_vals, pos_idx, neg_vals, neg_idx


## Running Intervention
Simply run the cell below which calls `intervener.generate_with_manipulation_sampling`, and observe the which tokens were most affected by amplifying the feature, and how the feature steers the model's output.


In [81]:

from intervention.intervener import Intervener

intervener = Intervener(act_generator.model, intervention_type='resid_post')

base_prompt = "I think that" # @param {type:"string"}

with torch.no_grad():
    base_logits = act_generator.model(act_generator.model.to_tokens(base_prompt))


feature_idx = 49 # @param {type:"integer"}
intervention_layer = 30 # @param {type:"integer"}
top_k = 10 # @param {type:"integer"}
alpha = 50 # @param {type:"integer"}
num_sentences_to_generate = 1 # @param {type:"integer"}

feature_vector = sae.decoder.weight[:, feature_idx]

intervened_logits = intervener.intervene(
                base_prompt,
                [feature_vector.to(device)],
                layers=[intervention_layer],
                alpha=alpha,

)

pos_vals, pos_idx, neg_vals, neg_idx = logit_diff(base_logits, intervened_logits, top_k)

print(f"Top {top_k} ↑ logit changes:")
for token_id, change in zip(pos_idx.tolist(), pos_vals.tolist()):
    token_str = _id_to_str(act_generator.model, token_id)
    print(f"  {token_str:>12}   {change:+.4f}")

print(f"\nTop {top_k} ↓ logit changes:")
for token_id, change in zip(neg_idx.tolist(), neg_vals.tolist()):
    token_str = _id_to_str(act_generator.model, token_id)
    print(f"  {token_str:>12}   {change:+.4f}")

delta = intervened_logits[0, -1, :] - base_logits[0, -1, :]

# steered generations (±alpha)
sentences_pos = intervener.generate_with_manipulation_sampling(
    base_prompt, [feature_vector], [layer],
    alpha=alpha, max_new_tokens=50, top_k=30, top_p=0.3,
    m=num_sentences_to_generate
)
sentences_neg = intervener.generate_with_manipulation_sampling(
    base_prompt, [feature_vector], [layer],
    alpha=-alpha, max_new_tokens=50, top_k=30, top_p=0.3,
    m=num_sentences_to_generate
)
print(f"Sentences positive: {sentences_pos}")
print(f"Sentences negative: {sentences_neg}")

Top 10 ↑ logit changes:
         solar   +2.8356
         Solar   +2.8310
         Solar   +2.6345
       eclipse   +2.6232
        eclips   +2.4870
        flares   +2.2765
         irrad   +2.2679
    telescopes   +2.1561
       Polaris   +2.1459
    satellites   +2.1392

Top 10 ↓ logit changes:
        Sloven   -1.5404
         Words   -1.4511
            Oo   -1.4048
       Wiggins   -1.3661
         itles   -1.3533
         Maple   -1.3234
          Lyme   -1.3204
        Breath   -1.3110
       Tolkien   -1.3104
     fragrance   -1.3084
Sentences positive: ["<|endoftext|>I think that's what this article will be about, but it should be pretty self-explanatory. There are many other reasons why your project will take longer than most, but I just want to touch on some areas that are often overlooked.\n\nA"]
Sentences negative: ['<|endoftext|>I think that there are certain places in the world where the more you know, the more you love. When I was living in Brazil, one of the most beau

########20########

Top 10 tokens:  and a the an, in to that as or

########49########

Top 10 tokens:  solarSolar eclipse eclips Solar telescopes flares telescope orbitsclipse



# The Challenge: Tricky Features

Feature 20 seems to be particularly elusive as it doesn't have any input activating tokens, and the vocabulary projections aren't informative!

🔍 Can you use other interpretability methods (e.g., Patchscopes) to see if you can uncover its functionality?

Copyright (c) 2025 Mor Geva and Daniela Gottesman